In [ ]:
%pip install --force-reinstall --no-deps git+https://github.com/chrisjcameron/TexSoup.git@mixed-args

In [1]:
import tarfile
import zipfile
import io
import os
import time
import math
import pickle
import itertools as itr
import collections as coll
import functools as ft
import regex as re
import chardet
from tqdm.auto import tqdm
from pytictoc import TicToc
import json

import pandas as pd
import numpy as np

import gcsfs
fs = gcsfs.GCSFileSystem()

from google.cloud import storage
from google.resumable_media.common import InvalidResponse
from google.cloud.exceptions import ClientError 

PROJECT_ID = "arxiv-development"
PRD_PROJECT = 'arxiv-production'
PRD_BUCKET_LOC = 'arxiv-production-data' 

from pylatexenc.latexwalker import LatexWalker, LatexEnvironmentNode, LatexGroupNode, LatexMacroNode, LatexCharsNode
from pylatexenc.latex2text import LatexNodes2Text


In [2]:
os.chdir("/home/jupyter/metadata-vertexai/")  # this needs to be the folder where notebook lives
import importlib
import phase_one_json as phase_one


In [3]:
#ror_rebuilt = phase_one.rorFinder(RECREATE_INDEX=True)

In [4]:
from IPython.core.interactiveshell import InteractiveShell
# pretty print all cell's output and not just the last one
InteractiveShell.ast_node_interactivity = "all"

In [5]:
def safe_divide(num, denom):
    return num / denom if denom != 0 else 0.0

In [6]:
import TexSoup as TS
from TexSoup.tokens import MATH_ENV_NAMES

In [7]:
new_def_str = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^}]*}\s*{[^}]*}
""".strip()

new_def_pat = re.compile(new_def_str)

In [8]:
new_def_pat.sub

<function Pattern.sub>

In [159]:
tail = '''
2310.01525v1 start
2312.15281v1 start
2302.08189v1 start
2304.06744v2 start
2307.03817v2 start
2306.03164v1 start
2312.14390v1 start
2311.07110v1 start
2301.08251v1 start
2312.10826v1 start
2310.01525v1 stop
2312.02605v1 start
2304.06744v2 stop
2312.03532v1 start
2306.03164v1 stop
2312.01252v1 start
2312.10826v1 stop
2304.04382v3 start
2311.07110v1 stop
2308.01385v1 start
2302.08189v1 stop
2305.06414v1 start
2312.02605v1 stop
2312.10979v2 start
2312.03532v1 stop
2306.01401v1 start
2301.08251v1 stop
2312.02955v1 start
2312.15281v1 stop
2309.09307v2 start
2304.04382v3 stop
2312.14390v1 stop
2305.06414v1 stop
2312.01252v1 stop
2312.10979v2 stop
2309.09307v2 stop
2312.02955v1 stop
2308.01385v1 stop
2307.03817v2 stop

'''.strip()

In [160]:
lines = sorted(x.split() for x in tail.splitlines())
cntr = coll.Counter(x[0] for x in lines)

In [161]:
cntr

Counter({'2301.08251v1': 2,
         '2302.08189v1': 2,
         '2304.04382v3': 2,
         '2304.06744v2': 2,
         '2305.06414v1': 2,
         '2306.03164v1': 2,
         '2307.03817v2': 2,
         '2308.01385v1': 2,
         '2309.09307v2': 2,
         '2310.01525v1': 2,
         '2311.07110v1': 2,
         '2312.01252v1': 2,
         '2312.02605v1': 2,
         '2312.02955v1': 2,
         '2312.03532v1': 2,
         '2312.10826v1': 2,
         '2312.10979v2': 2,
         '2312.14390v1': 2,
         '2312.15281v1': 2,
         '2306.01401v1': 1})

## Test TexSoup

In [ ]:
test_ids_df = pd.read_csv("gs://institutional-extract-scratch/reference/arx_ids/2311_ids.csv")
test_ids_df.head()

In [ ]:
sample_size = 1000
batch_size = 20
parallel_workers = 8
thread_workers = 10
#import concurrent.futures
#import phase_one

def test_id(arx_id):
    yymm = arx_id.split(".")[0]
    paper_id = arx_id.split("v")[0]
    tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"

    res = None
    try:
        tar_bytes = phase_one.bytes_from_tarpath(tar_path)
        candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
        source_text = phase_one.source_from_archive(tar_bytes, candidate_files[0])
        tsoup = TS.TexSoup(source_text, tolerance=0)
        return 1
    except:
        try:
            tsoup = TS.TexSoup(source_text, tolerance=1)
            return 2
        except:
            return 0
    return 0

res = []
for arx_id in tqdm(test_ids_df['arx_id'].iloc[0:100]):
    res.append(test_id(arx_id))
    
coll.Counter(res)

In [12]:
%%time
arx_id = '2310.03838v1'
phase_one.get_single_file_results(arx_id, verbose=True, vverbose=True)

Processing ftp/arxiv/papers/2310/2310.03838.tar.gz
	Processing ftp/arxiv/papers/2310/2310.03838.tar.gz, main.tex



KeyboardInterrupt



In [162]:
%%time
arx_id = '2306.01401v1'

yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"


tar_bytes = phase_one.bytes_from_tarpath(tar_path)
candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
tex_main = candidate_files[0]
inc_list = []
if include_dict: inc_list = include_dict.get(tex_main, [])
print(f"Got main: {tex_main} with {inc_list}")

for c_file in candidate_files:
    print(f"Starting: {c_file}")
    res_gen = phase_one.extract_pre_abstract_content(tar_bytes, tex_main=c_file, include_list=None, file_path=tar_path)
    res_list = list(res_gen)
    print(f"finished with {c_file}, got {len(res_list)}")

Got main: main.tex with ['packages_article.tex', 'symbols_env.tex', 'symbols.tex', 'abstractShort.tex', 'introShort.tex', 'prelims.tex', 'BA.tex', 'ICSig.tex', 'statVSS.tex', 'PVMT.tex', 'statMVSS.tex', 'Triples.tex', 'mpc.tex', 'Impossibility.tex', 'Acknowledgements.tex', 'AppBA.tex', 'AppICP.tex', 'AppVSS.tex', 'AppRec.tex', 'AppMDVSS.tex', 'AppTriples.tex', 'AppMPC.tex']
Starting: main.tex
finished with main.tex, got 4
Starting: Acknowledgements.tex
finished with Acknowledgements.tex, got 1
Starting: AppBA.tex
finished with AppBA.tex, got 1
Starting: AppICP.tex
finished with AppICP.tex, got 1
Starting: AppMDVSS.tex
finished with AppMDVSS.tex, got 1
Starting: AppMPC.tex
finished with AppMPC.tex, got 1
Starting: AppRec.tex
finished with AppRec.tex, got 1
Starting: AppTriples.tex
finished with AppTriples.tex, got 1
Starting: AppVSS.tex
finished with AppVSS.tex, got 1
Starting: BA.tex
finished with BA.tex, got 1
Starting: ICSig.tex
finished with ICSig.tex, got 1
Starting: Impossibility.


KeyboardInterrupt



In [165]:
arx_id = '2310.03838v1'  # actually fails #'symbols.tex'
arx_id = '2306.01401v1'  # symbols_env.tex

def append_node_contents(focus_nodes, full_nodelist, result_list):
    for i,node in focus_nodes:
        result_list.append(node.latex_verbatim())
        try:
            idx_plus = 1
            while True:
                if idx_plus > 10:
                    break
                follow_node = full_nodelist[i+idx_plus]
                if isinstance(follow_node, LatexGroupNode):
                    result_list.append(follow_node.latex_verbatim())
                    break
                idx_plus += 1
        except IndexError:
            pass

yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"


tar_bytes = phase_one.bytes_from_tarpath(tar_path)
candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
tex_main = candidate_files[0]
include_list = None #[]
#if include_dict: include_list = include_dict.get(tex_main, [])

incl_res_gen_list = []
if include_list:
    for inc_file in include_list:
        incl_res_gen_list.append(extract_pre_abstract_content(tar_bytes, tex_main=inc_file, file_path=file_path))

tex_main = "symbols_env.tex"  #
print(f"Got main: {tex_main}")
if include_dict:
    print(f"Got included_files: {include_dict.get('tex_main')}")

doc = """
Parses a .tex file:
- Removes LaTeX comments
- Extracts institution names (via recursive regex)
- Extracts text before the abstract
"""

source_text = phase_one.source_from_archive(tar_bytes, tex_main)

# Remove LaTeX comments (lines starting with non-escaped %)
new_def_str = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^}]*}\s*{[^}]*}
""".strip()
new_def_v1 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*{[^{}]*}
""".strip()
new_def_v2 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*(\[[^\]]*\])?\s*{[^{}]*}
""".strip()
new_def_v3 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*(\[[^\]]*\])?\s*{(?:[^{}]*|{[^{}]*})*}\s*{(?:[^{}]*|{[^{}]*})*}
""".strip()

strip_env = r"""
\\newenvironment\{[^\}]+\}\s*\{\s*((?>[^{}]+|\{(?1)\})*)\}\s*\{\s*((?>[^{}]+|\{(?1)\})*)\}
""".strip()

new_def_v1_pat = re.compile(new_def_v1, re.DOTALL, re.MULTILINE)
new_def_v2_pat = re.compile(new_def_v2, re.DOTALL, re.MULTILINE)
new_def_v3_pat = re.compile(new_def_v3, re.DOTALL, re.MULTILINE)
strip_env_pat  = re.compile(strip_env, re.DOTALL)
content = re.sub(r"(?<!\\)%.*", "", source_text)
content = strip_env_pat.sub("\n", content)
content = new_def_v3_pat.sub("\n", content)
#content = new_def_v1_pat.sub("\n", content)
#content = new_def_v2_pat.sub("\n", content)
#res_list = []

# try parsing latex:
auth_macros = set([
    "author", "auth", "authors",
    "institute", "inst", "institution",
    "affiliation", "affil", "affiliations",
    "address",
    "cmsinstitute",
])
supstr = set([
    "\\textsuperscript",
])
latex_extracted_institutions = []

lxwkr = LatexWalker(content, tolerant_parsing=True)
print("Starting get nodes main")
(nodelist, pos, len_) = lxwkr.get_latex_nodes(read_max_nodes=2000) #789
print("Stopping get nodes main")
focus_nodes = [
  (i,node) for i,node in enumerate(nodelist)
  if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
]
len(nodelist)
nodelist[-3:]
#candidate_files

Got main: symbols_env.tex
Got included_files: None
Starting get nodes main
Stopping get nodes main


7

[LatexGroupNode(parsing_state=<parsing state 139662874957744>, pos=50, len=10, nodelist=[LatexCharsNode(parsing_state=<parsing state 139662874957744>, pos=51, len=8, chars='titlebox')], delimiters=('{', '}')),
 LatexCharsNode(parsing_state=<parsing state 139662874957744>, pos=60, len=4, chars='[5]\n'),
 LatexGroupNode(parsing_state=<parsing state 139662874957744>, pos=64, len=4229, nodelist=[LatexMacroNode(parsing_state=<parsing state 139662874957744>, pos=65, len=9, macroname='mdfsetup', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexGroupNode(parsing_state=<parsing state 139662874957744>, pos=74, len=278, nodelist=[LatexCharsNode(parsing_state=<parsing state 139662874957744>, pos=75, len=33, chars='\n\t\tstyle=#2,\n\t\tinnertopmargin=1.1'), LatexMacroNode(parsing_state=<parsing state 139662874957744>, pos=108, len=13, macroname='baselineskip', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexCharsNode(parsing_state=<parsing 

In [154]:
initial_def = r"""
\\newenvironment\{[^\}]+\}\s*\{\s*((?>[^{}]+|\{(?1)\})*)\}\s*\{\s*((?>[^{}]+|\{(?1)\})*)\}
""".strip()
args = r"""
\s*(?:\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}\s*
""".strip()
rest = r"""
\s*{[^{}]*}\s*(\[[^\]]*\])?\s*{(?:[^{}]*|{[^{}]*})*}\s*{(?:[^{}]*|{[^{}]*})*}
""".strip()
new_def_v3_pat = re.compile(initial_def, re.DOTALL)
new_def_v3_pat.search(content[26365:26365+284])

<regex.Match object; span=(0, 281), match='\\newenvironment{compactlist}{\n\t\\begin{list}{{$\\bullet$}}{\n\t\t\t\\setlength\\partopsep{0pt}\n\t\t\t\\setlength\\parskip{0pt}\n\t\t\t\\setlength\\parsep{0pt}\n\t\t\t\\setlength\\topsep{0pt}\n\t\t\t\\setlength\\itemsep{0pt}\n\t\t\t\\setlength{\\itemindent}{0.4pt}\n\t\t\t\\setlength{\\leftmargin}{10pt}\n\t\t}\n\t}{\n\t\\end{list}\n}'>

In [151]:
print(content[26365:26365+266])

\newenvironment{compactlist}{
	\begin{list}{{$\bullet$}}{
			\setlength\partopsep{0pt}
			\setlength\parskip{0pt}
			\setlength\parsep{0pt}
			\setlength\topsep{0pt}
			\setlength\itemsep{0pt}
			\setlength{\itemindent}{0.4pt}
			\setlength{\leftmargin}{10pt}
		}
	}


In [22]:
arx_id = '2310.03838v1'  # actually fails



def append_node_contents(focus_nodes, full_nodelist, result_list):
    for i,node in focus_nodes:
        result_list.append(node.latex_verbatim())
        try:
            idx_plus = 1
            while True:
                if idx_plus > 10:
                    break
                follow_node = full_nodelist[i+idx_plus]
                if isinstance(follow_node, LatexGroupNode):
                    result_list.append(follow_node.latex_verbatim())
                    break
                idx_plus += 1
        except IndexError:
            pass

yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"


tar_bytes = phase_one.bytes_from_tarpath(tar_path)
candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
tex_main = 'symbols.tex' #candidate_files[0]

include_list = None #[]
if include_list: include_dict.get(tex_main, [])

incl_res_gen_list = []
if include_list:
    for inc_file in include_list:
        incl_res_gen_list.append(phase_one.extract_pre_abstract_content(tar_bytes, tex_main=inc_file, file_path=file_path))

print(f"Got main: {tex_main}")
if include_dict:
    print(f"Got included_files: {include_dict.get('tex_main')}")

doc = """
Parses a .tex file:
- Removes LaTeX comments
- Extracts institution names (via recursive regex)
- Extracts text before the abstract
"""

source_text = phase_one.source_from_archive(tar_bytes, tex_main)

# Remove LaTeX comments (lines starting with non-escaped %)
new_def_str = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^}]*}\s*{[^}]*}
""".strip()
new_def_pat = re.compile(new_def_str)
content = re.sub(r"(?<!\\)%.*", "", source_text)
content = new_def_pat.sub("\n", content)
del source_text
#res_list = []

# try parsing latex:
# Note: names are lowered before compare
auth_macros = set([
    "author", "auth", "authors",
    "institute", "inst", "institution",
    "university",
    "orgname",
    "affiliation", "affil", "affiliations", "aff",
    "address",
    "cmsinstitute", "icmlaffiliation",
])
supstr = set([
    "\\textsuperscript",
])
latex_extracted_institutions = []
for inc_gen in incl_res_gen_list:
    latex_extracted_institutions.append(next(inc_gen))

try:
    lxwkr = LatexWalker(content, tolerant_parsing=True)
    (nodelist, pos, len_) = lxwkr.get_latex_nodes()
    focus_nodes = [
      (i,node) for i,node in enumerate(nodelist)
      if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
    ]
    if focus_nodes:
        append_node_contents(focus_nodes, nodelist, latex_extracted_institutions)
        # Get /textsuperscript contents if indicated
        # @todo: also get the $^[1]$Institution style indicators 
        if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
            sup_res = extract_texsuperscript(nodelist)
            latex_extracted_institutions.extend(sup_res)
    else:
        doc = [
            node for node in nodelist
            if isinstance(node, LatexEnvironmentNode) and node.environmentname=='document'
        ]
        if doc:
            docnodelist = doc[0].nodelist
            focus_doc_nodes = [
              (i,node) for i, node in enumerate(docnodelist)
              if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
            ]
            append_node_contents(focus_doc_nodes, docnodelist, latex_extracted_institutions)
            # Get /textsuperscript contents if indicated
            # @todo: also get the $^[1]$Institution style indicators 
            if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
                sup_res = extract_texsuperscript(docnodelist)
                latex_extracted_institutions.extend(sup_res)
    #for var in ('nodelist', 'pos', 'len_', 'sup_res', 'focus_nodes', 'doc', 'docnodelist', 'focus_doc_nodes'):
    #    if var in locals(): del locals()[var]
    if latex_extracted_institutions:
        #res_list.append(latex_extracted_institutions)
        #yield "\n".join(latex_extracted_institutions)
        pass
except Exception as e:
    print(f"\nOverly broad except in extract_pre_abstract_content(): {e} for {file_path}-{tex_main}")
    pass

len(nodelist)
latex_extracted_institutions

Got main: main.tex
Got included_files: None


28

['\\author{Harsh Chaudhari, Giorgio Severi, Alina Oprea, Jonathan Ullman \n\n\n\n\\\\\nKhoury College of Computer Science\\\\\nNortheastern University\\\\\n\n\\texttt{\\{chaudhari.ha, severi.g, a.oprea, j.ullman\\}@northeastern.edu} \\\\\n}']

In [16]:
nodelist[-2]

LatexEnvironmentNode(parsing_state=<parsing state 139663144504528>, pos=412, len=4346, environmentname='document', nodelist=[LatexCharsNode(parsing_state=<parsing state 139663144504528>, pos=428, len=2, chars='\n\n'), LatexMacroNode(parsing_state=<parsing state 139663144504528>, pos=430, len=83, macroname='title', nodeargd=ParsedMacroArgs(argspec='{', argnlist=[LatexGroupNode(parsing_state=<parsing state 139663144504528>, pos=436, len=77, nodelist=[LatexCharsNode(parsing_state=<parsing state 139663144504528>, pos=437, len=75, chars='Chameleon: Increasing Label-Only Membership Leakage with Adaptive Poisoning')], delimiters=('{', '}'))]), macro_post_space=''), LatexCharsNode(parsing_state=<parsing state 139663144504528>, pos=513, len=2, chars='\n\n'), LatexMacroNode(parsing_state=<parsing state 139663144504528>, pos=515, len=217, macroname='author', nodeargd=ParsedMacroArgs(argspec='{', argnlist=[LatexGroupNode(parsing_state=<parsing state 139663144504528>, pos=522, len=210, nodelist=[La

In [ ]:
%%time 
arx_id = '2301.07517v1'
yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"
tar_bytes = phase_one.bytes_from_tarpath(tar_path)

candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
source_text = phase_one.source_from_archive(tar_bytes, candidate_files[0])
tsoup = TS.TexSoup(source_text, tolerance=0)